**Author**: Ortega Rivera Leonardo Fabyan

<div style="margin: 0 auto; width: 80%; text-align: justify; padding: 20px; font-style: italic;">

**Abstract.** This work presents a baseline approach to the classification of dark matter substructure in gravitational lensing images using a modified ResNet-18 architecture trained from scratch on synthetically generated data. The dataset comprises 37,500 single-channel images of resolution $150 \times 150$ pixels, partitioned into three balanced classes - No Substructure, SubHalo (Sphere), and Vortex - and generated using the $\texttt{lenstronomy}$ simulation framework. The architecture was adapted to accept single-channel input and produce three-class logit outputs, and trained using the AdamW optimizer with hyperparameters selected via Bayesian optimization through Optuna. Prior to full-scale training, a structured suite of six sanity checks was conducted to validate the data pipeline, architectural integrity, and optimization stack. The model was trained for 100 epochs, achieving a validation accuracy of 93.76% and a macro-averaged F1-score of $0.9370$ at the best checkpoint. Evaluation on the held-out validation set yields a mean AUC of $0.9907$ across all classes under the One-vs-Rest ROC scheme, with the Vortex class presenting the greatest discriminative challenge. These results establish a strong and reproducible baseline for subsequent experimentation with more expressive architectures.

</div>

In [1]:
import sys
import os

sys.path.append(os.path.abspath('../src'))

from dataprocessing.dataloader import build_dataloaders # type: ignore
from utils.seed import seed_everything # type: ignore

DIR: str = 'task1_images'
WEIGHTS_NAME: str = 'model-1'

os.makedirs(DIR, exist_ok=True)
os.makedirs('weights', exist_ok=True)

def format_fig(fig) -> str:
    title = fig.layout.title.text
    return DIR + '/' + title.replace('.', '').replace(' ', '_').lower() + '.png'

seed_everything(42)

# Dataset

The dataset comprises 37,500 single-channel gravitational lensing images of spatial resolution $150 \times 150$ pixels, partitioned into a training split of 30,000 samples and a validation split of 7,500 samples, following an 80/20 ratio. Each image is assigned to one of three mutually exclusive classes: No Substructure, SubHalo (Sphere), and Vortex. The class distribution is perfectly balanced across both splits, with each category contributing equally to the total sample count (10,000 training and 2,500 validation samples per class), thereby eliminating class imbalance as a confounding factor in model evaluation. All images were generated synthetically using the $\texttt{lenstronomy}$ Python package, a well-established gravitational lensing simulation framework. A summary of the dataset composition is provided in Table 1. 
More information about the dataset, see [EDA Notebook](./Exploratory%20Analysis%20and%20Assessment%20of%20Spatial%20Separability%20in%20Dark%20Matter%20Substructure%20Dataset.pdf).

\begin{table}[H]
\centering
\begin{tabular}{|l|l|l|l|l|l|}
\hline
\textbf{Split} & \textbf{Total Samples} & \textbf{\% of Dataset} & \textbf{Class: No} & \textbf{Class: Sphere} & \textbf{Class: Vort} \\ \hline
Train & 30,000 & 80 \% & 10,000 & 10,000 & 10,000 \\ \hline
Validation & 7,500 & 20 \% & 2,500 & 2,500 & 2,500 \\ \hline
\end{tabular}
\\ Table 1: Dataset summary.
\end{table}

In [2]:
# add tag: hide_cell
train_loader, valid_loader = build_dataloaders('../dataset', 64, 128, 4)

split: train
	class: no (label: 0), number of samples: 10000
	class: sphere (label: 1), number of samples: 10000
	class: vort (label: 2), number of samples: 10000
	total samples: 30000
split: val
	class: no (label: 0), number of samples: 2500
	class: sphere (label: 1), number of samples: 2500
	class: vort (label: 2), number of samples: 2500
	total samples: 7500


In [3]:
from plotly.subplots import make_subplots
from plotly import express as px
import numpy as np

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('No Substructure',
                    'SubHalo (Sphere)',
                    'Vortex'))

img = np.load('../dataset/train/no/10.npy').squeeze()
sub_fig = px.imshow(img)
fig.add_trace(sub_fig.data[0], row=1, col=1)

img = np.load('../dataset/train/sphere/20.npy').squeeze()
sub_fig = px.imshow(img)
fig.add_trace(sub_fig.data[0], row=1, col=2)

img = np.load('../dataset/train/vort/30.npy').squeeze()
sub_fig = px.imshow(img)
fig.add_trace(sub_fig.data[0], row=1, col=3)

fig.write_image(fr'{DIR}/classes.png',
    width=1048, height=450)
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=1\textwidth]{task1_images/classes.png}
Figure 1: Example of a No Substructure, a Sphere, and a Vortex image.
\end{figure}

# Model

In [4]:
# add tag: hide_cell
import torch
import torch.nn as nn
import torchvision.models as models

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
device

'cuda'

## Architecture used

ResNet-18 is selected as the base architecture on account of its favorable trade-off between representational capacity and computational efficiency. Relative to deeper variants such as ResNet-50 or ResNet-152, ResNet-18 contains significantly fewer parameters, which reduces the risk of overfitting on moderately sized datasets while enabling faster experimentation and iteration. Furthermore, the residual learning framework inherently mitigates vanishing gradient issues, ensuring stable optimization even in early experimental stages. More complex architectures, such as DenseNet, introduce higher memory and computational overhead without a guaranteed performance gain for this particular task. Given the exploratory nature of this work and the controlled dataset size, ResNet-18 constitutes a robust and well-understood baseline for validating the training pipeline prior to scaling toward more expressive models.
The architecture is trained from scratch, without pre-trained weights, and two targeted modifications are introduced to adapt it to the present single-channel, three-class classification problem. First, the initial convolutional layer (Conv1) is modified to accept single-channel input by reducing its in-channel dimension from 3 to 1. Second, the final fully connected layer is replaced to produce a three-dimensional output vector, mapping the learned representations to the three target classes: No Substructure, SubHalo (Sphere), and Vortex.

\begin{figure}[h]
\centering
\includegraphics[width=0.6\textwidth]{task1_images/resnet18-90.png}
\\ Figure 2: Original ResNet-18 Architecture. Taken from Dive into Deep Learning, Chapter 8.6: “Residual Networks (ResNet) and ResNeXt”.
\end{figure}

In [5]:
C_CLASSES: int = 3

In [6]:
def init_model():
    # initilizated base ResNet18 model, no pre-trained weights are used
    model = models.resnet18(weights=None)

    # edit the first conv layer (conv1) to accept one channel as input
    model.conv1 = nn.Conv2d(
        in_channels=1,
        out_channels=model.conv1.out_channels,
        kernel_size=model.conv1.kernel_size[0],
        stride=model.conv1.stride[0],
        padding=model.conv1.padding[0],
        bias=False,
    )

    # edit the last layer (Full Connected)
    model.fc = nn.Linear(
        in_features=model.fc.in_features,
        out_features=C_CLASSES, # <- output
    )

    return model.to(device)

model = init_model()

# We don't need to add an activation layer such as Softmax at the end of the model 
# because PyTorch's CrossEntropyLoss already applies this activation to the input.

In [7]:
def init_criterion():
    return nn.CrossEntropyLoss()

In [8]:
import torch.optim as optim

def init_optimizer(parameters, lr: float, wd: None | float = None):
    if wd is None:
        return optim.Adam(parameters, lr=lr)
    return optim.AdamW(params=parameters, 
        lr=lr, weight_decay=wd)

## Sanity checks

Prior to any hyperparameter search or full-scale training, a structured suite of six diagnostic checks was conducted to verify the correctness of the data pipeline, architecture, and optimization stack. This practice, advocated by Karpathy (2019) in the context of neural network training methodology, ensures that potential implementation errors are identified and resolved before they can silently corrupt the training process or lead to misleading results. The checks are ordered by increasing proximity to the optimization loop: beginning with the data loading stage (Sections 2.2.1-2.2.2), proceeding through architectural integrity (Section 2.2.3), and culminating in optimization-level diagnostics (Sections 2.2.4-2.2.6). A failure at any of these stages would have been treated as a blocking issue requiring resolution before proceeding.

### Batch Variance Verification

In stochastic optimization, mini-batches are assumed to constitute independent and identically distributed (i.i.d.) samples from the underlying data distribution. Violations of this assumption - such as duplicated batches, defective shuffling, or temporal correlations in the data loading pipeline - can introduce systematic bias in gradient estimates and consequently degrade convergence.
To empirically validate that the DataLoader produces sufficiently distinct batches, four consecutive batches were sampled from the training set and their pairwise dissimilarity was quantified via the Mean Squared Error (MSE). The resulting similarity matrix exhibits non-zero off-diagonal entries, confirming that inter-batch variance is present and that no temporal redundancies are introduced during data loading. The diagonal entries are identically zero, as expected for self-comparisons.

In [9]:
samples = []

for i, (sample, label) in enumerate(train_loader):
    samples.append(sample)
    
    if i == 3:
        break
    
mse = nn.MSELoss()
mse_values = np.full(shape=2 * (len(samples),), fill_value=np.nan)

for i in range(len(samples)):
    for j in range(i, len(samples)):
        mse_values[i, j] = mse(samples[i], samples[j]).item()
        
fig = px.imshow(
    mse_values,
    text_auto='.4f', # type: ignore
    labels=dict(x='Batch Index', y='Batch Index', color='MSE'),
    title='MSE between pair of batches in Train set',
    height=524, width=524,
)

fig.write_image(format_fig(fig))
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=0.4\textwidth]{task1_images/mse_between_pair_of_batches_in_train_set}
\\ Figure 3: MSE between 4 pair of batches in Train set.
\end{figure}

### Augmentation Invariance Validation

Data augmentation is employed to enrich the effective training distribution and improve generalization; however, in physically grounded domains, geometric transformations may inadvertently violate the invariances of the underlying phenomenon, thereby introducing label noise. It is therefore essential to verify that the augmentation pipeline preserves class semantics prior to training.
The augmentation strategy applied consists of random horizontal flips ($p = 0.5$) and random rotations uniformly sampled from the interval [−90°, 90°]. The assumption of rotational invariance up to 90° is physically motivated but must ultimately be validated against domain knowledge. Prior to augmentation, all images are normalized via min-max normalization to ensure a consistent pixel intensity range across the dataset.
Figure 4 presents a visual inspection of four representative samples alongside three independently drawn augmented variants per sample. The qualitative consistency between original and transformed images supports the hypothesis that the selected transformations do not alter the discriminative morphological features associated with each class.

In [10]:
from torchvision.transforms import v2

transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=[-90, 90])
])

In [11]:
from plotly.subplots import make_subplots

n_samples: int = 4

fig = make_subplots(rows=n_samples, cols=n_samples,
    column_titles=[f'Sample #{i}' for i in range(1, n_samples + 1)], 
    row_titles=['Original'] + [f'Trans #{i}' for i in range(1, n_samples)])

for i in range(n_samples):
    for j in range(n_samples):
        if i == 0:
            sub_fig = px.imshow(samples[0][j].squeeze())
        else:
            img_trans = transforms(samples[0][j]).squeeze()
            sub_fig = px.imshow(img_trans)
        fig.add_trace(sub_fig.data[0], row=i + 1, col=j + 1)

fig.update_layout(height=786, width=786, title_text='Data Augmentation')

fig.write_image(format_fig(fig))
fig.show()

\begin{figure}[H]
\centering
\includegraphics[width=0.8\textwidth]{task1_images/data_augmentation.png}
\\ Figure 4: Example of 4 random images with 3 random Data Augmentation.
\end{figure}

### Forward Pass Integrity Check

Modifying a standard architecture to accommodate a different input modality and output cardinality introduces the risk of dimensional mismatches or silent broadcasting errors that may not surface until training. This step empirically verifies that the adapted ResNet-18 correctly maps input tensors to the logit space end-to-end.
A single batch from the training DataLoader was passed through the model in inference mode, and the shape of the resulting output tensor was inspected. The obtained output shape of (64, 3) is consistent with the expected behavior: the batch dimension of 64 matches the configured DataLoader batch size, and the output dimension of 3 corresponds to the number of target classes. This confirms that the architectural modifications - reduction of Conv1 to a single input channel and replacement of the final fully connected layer with a three-unit output - were applied correctly, with no broadcasting errors or dimensional mismatches along the forward pass.

In [12]:
# add tag: hide_cell
model.eval()
with torch.no_grad():
    samples, labels = next(iter(train_loader))
    samples = samples.to(device)
    labels = labels.to(device)
    
    pred = model(samples)
    print(f'Model output shape: {pred.shape}')

Model output shape: torch.Size([64, 3])


### Entropy-Based Baseline Loss Validation

Prior to any parameter updates, a correctly implemented classifier initialized with random weights should produce output logits that approximate a uniform distribution over all classes. Under this condition, the expected cross-entropy loss corresponds to the entropy of the uniform distribution over $C$ classes
$$
\mathcal{L} = - \ln \left(\frac{1}{C} \right) = \ln \left(C \right)
$$
For $C = 3$, this yields a theoretical baseline of $\ln(3) \approx 1.0986$. Deviations from this value are expected to be small and attributable to random initialization variance.
To empirically validate this, the cross-entropy loss was computed over the full training set using the randomly initialized model in inference mode. The observed mean loss across all batches was $1.1019$, representing a relative deviation of approximately $0.003 \%$ above the theoretical baseline. This result is well within the expected range of stochastic variation and confirms that the loss function, the output layer, and the class distribution are all correctly configured before training begins.


In [13]:
# add tag: hide_cell
criterion = init_criterion()
ce_loss_total: float = 0.0
n_batches: int = 0

model.eval()
with torch.no_grad():
    for samples, labels in train_loader:
        samples = samples.to(device)
        labels = labels.to(device)
        
        pred = model(samples)
        
        n_batches += 1
        ce_loss_total += criterion(pred, labels).item()
        
ce_loss_total /= n_batches
print(f'Loss total over train set: {ce_loss_total:.4f}')

Loss total over train set: 1.1019


### Single-Batch Memorization Test

A sufficiently expressive model equipped with a correctly implemented optimization pipeline should be capable of memorizing a small, fixed dataset - that is, driving the training loss arbitrarily close to zero. This constitutes the most critical sanity check of the pipeline, as successful overfitting on a single batch provides simultaneous evidence that the architecture, the loss function, gradient backpropagation, and the optimizer are all functioning coherently and in concert.

To perform this verification, a single batch of 64 samples was held fixed and the model was trained on it exclusively for 200 epochs using the Adam optimizer with a learning rate of $1 \times 10^{-3}$ and cross-entropy loss. Anomaly detection was enabled via $\texttt{torch.autograd.detect\_anomaly()}$ to surface any numerical instabilities during backpropagation. The training loss decreased monotonically and converged to $0.0$ by epoch 200, confirming that the optimization pipeline is correctly implemented and that the model possesses sufficient capacity to overfit a single batch without any detected anomalies.

In [14]:
# add tag: hide_cell
model.train()

# select one batch
samples, labels = next(iter(train_loader))
samples = samples.to(device)
labels = labels.to(device)

epochs: int = 200
optimizer = init_optimizer(model.parameters(), 0.001)

for epoch in range(1, epochs + 1):
    try:
        with torch.autograd.detect_anomaly():
            pred = model(samples)
    
            loss = criterion(pred, labels)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    except RuntimeError as e:
        print(f'Anomaly detected at epoch [{epoch}]: {e}')
        break
    
    if epoch % 10 == 0:
        print(f'Epoch [{epoch}/{epochs}], Loss: {loss.item():.4f}')

/tmp/ipykernel_136912/1505916730.py:14: UserWarning: Anomaly Detection has been enabled. This mode will increase the runtime and should only be enabled for debugging.
  with torch.autograd.detect_anomaly():


Epoch [10/200], Loss: 0.0052
Epoch [20/200], Loss: 0.0008
Epoch [30/200], Loss: 0.0003
Epoch [40/200], Loss: 0.0002
Epoch [50/200], Loss: 0.0001
Epoch [60/200], Loss: 0.0001
Epoch [70/200], Loss: 0.0001
Epoch [80/200], Loss: 0.0001
Epoch [90/200], Loss: 0.0001
Epoch [100/200], Loss: 0.0000
Epoch [110/200], Loss: 0.0000
Epoch [120/200], Loss: 0.0000
Epoch [130/200], Loss: 0.0000
Epoch [140/200], Loss: 0.0000
Epoch [150/200], Loss: 0.0000
Epoch [160/200], Loss: 0.0000
Epoch [170/200], Loss: 0.0000
Epoch [180/200], Loss: 0.0000
Epoch [190/200], Loss: 0.0000
Epoch [200/200], Loss: 0.0000


### Gradient Stability and Numerical Integrity Check

Monitoring gradient statistics throughout the network is essential for detecting vanishing gradients, exploding gradients, or numerical instabilities such as NaN or Inf values, all of which can silently corrupt the optimization process. The first and last layers serve as diagnostic anchors: if the gradient of the final layer has vanished or exploded, this pathology will propagate backwards through the entire network; conversely, if the gradient of the first layer is already degenerate, it is highly likely that all intermediate layers share the same condition.

Following the single-batch overfitting experiment, the gradients of the final fully connected layer (FC) and the first convolutional layer (Conv1) were inspected for numerical integrity and magnitude. The results are summarized as follows:

+ FC layer: 0 NaN, 0 Inf, mean absolute gradient $\approx 1.12 \times 10^{-5}$
+ Conv1 layer: 0 NaN, 0 Inf, mean absolute gradient $\approx 5.96 \times 10^{-7}$

The absence of NaN and Inf values confirms numerical stability throughout the forward and backward passes. The observed gradient magnitudes are small but non-zero, which is expected given that the model has already converged on the single batch by epoch 200. The attenuation from the last layer to the first layer is consistent with standard backpropagation behavior through a network of this depth and does not indicate a vanishing gradient pathology in the context of this verification.

In [15]:
# add tag: hide_cell
def check_nan(x: torch.Tensor | None) -> None:
    if x is None:
        print(f'  Received a None')
        return
    
    print(f'  Total NaN: {torch.isnan(x).sum().item()}')
    print(f'  Total Inf: {torch.isinf(x).sum().item()}')
    print(f'  Avg Absolute gradient value: {x.abs().mean().item()}')
    
print('Last layer (FC) grad checking:')
check_nan(model.fc.weight.grad)
print('- ' * 5)
print('First layer (Conv1) grad checking:')
check_nan(model.conv1.weight.grad)

Last layer (FC) grad checking:
  Total NaN: 0
  Total Inf: 0
  Avg Absolute gradient value: 1.1234379599045496e-05
- - - - - 
First layer (Conv1) grad checking:
  Total NaN: 0
  Total Inf: 0
  Avg Absolute gradient value: 5.960292241979914e-07


## Hyperparameter optimization

The two primary optimization hyperparameters - learning rate and weight decay ($L_{2}$ regularization) - were selected via automated hyperparameter search using the $\texttt{Optuna}$ framework with the Tree-structured Parzen Estimator (TPE) sampler. Both hyperparameters were searched over a logarithmically spaced range of $[10^{-5}, 10^{-1}]$, reflecting the several orders of magnitude over which these values can meaningfully affect convergence behavior.

Each trial consisted of training a freshly initialized model for 8 epochs on the full training set using the AdamW optimizer, with data augmentation applied inline during training. The objective metric was the mean cross-entropy loss evaluated on the validation set at the end of each trial. A total of 25 trials were conducted, with a fixed random seed of 42 applied to both the sampler and model initialization to ensure reproducibility.
The optimal hyperparameter configuration identified by the search is as follows:
$1.0200 \times 10^{-4}$ for learning rate, 
$1.3699 \times 10^{-4}$ for weight decay.
These values were subsequently used for full-scale model training, as described in Section 3.

In [ ]:
# add tag: hide_cell
import optuna

In [ ]:
def objective(trial) -> float:
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True) # learning rate
    wd = trial.suggest_float('wd', 1e-5, 1e-1, log=True) # weight decay (L2)
    
    seed_everything(42)
    model = init_model()
    
    criterion = init_criterion()
    optimizer = init_optimizer(model.parameters(),
        lr=lr, wd=wd)
    
    model.train()
    for epoch in range(8):
        for samples, labels in train_loader:
            samples = samples.to(device)
            labels = labels.to(device)
            
            samples = transforms(samples)
            pred = model(samples)
    
            loss = criterion(pred, labels)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
    model.eval()
    with torch.no_grad():
        ce_loss_total: float = 0.0
        n_batches = 0
        
        for samples, labels in valid_loader:
            samples = samples.to(device)
            labels = labels.to(device)
            
            pred = model(samples)
            
            n_batches += 1
            ce_loss_total += criterion(pred, labels).item()
            
    return ce_loss_total / n_batches

In [16]:
for var_name in ['samples', 'labels', 'model', 'criterion', 'optimizer']:
    if var_name in globals():
        del globals()[var_name]

torch.cuda.empty_cache()

In [17]:
# add tag: hide_cell
import gc
gc.collect()

50874

In [ ]:
# add tag: hide_cell
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='minimize', sampler=sampler)

study.optimize(objective, n_trials=25)

# Training

The model was trained for 100 epochs on the full training set using the AdamW optimizer with the hyperparameters identified in Section 2.3: a learning rate of $1.0200 \times 10^{-4}$ and a weight decay of $1.3699 \times 10^{-4}$. Data augmentation was applied inline during training as described in Section 2.2.2. All experiments were conducted on an NVIDIA GeForce RTX 2070 SUPER, with the total training duration amounting to approximately 1 hour and 16 minutes.

Training and validation metrics - cross-entropy loss, macro-averaged F1-score, and accuracy - were logged at every epoch via Weights & Biases. The model checkpoint corresponding to the lowest validation loss observed during training was saved and used for all subsequent evaluation. The best checkpoint was obtained at epoch 86, yielding the following metrics:

\begin{table}[H]
\centering
\begin{tabular}{|l|l|l|l|}
\hline
\textbf{Split} & \textbf{Loss} & \textbf{Accuracy} & \textbf{Macro F1} \\ \hline
Train & 0.1609 & 0.9411 \% & 0.9409 \\ \hline
Validation & 0.1720 & 0.9376 \% & 0.9370 \\ \hline
\end{tabular}
\end{table}

The learning curves depicted in Figure 5 show a consistent decrease in training loss alongside stable validation performance, with no evidence of severe overfitting.

In [ ]:
# add tag: hide_cell
import wandb

config = {
    'learning_rate': study.best_params['lr'],
    'weight_decay': study.best_params['wd'],
    'architecture': 'ResNet18-Modded',
    'optimizer': 'AdamW',
    'dataset': 'BlackMatterSubstructure-Classification',
    'epochs': 100,
}

run = wandb.init(
    entity="leofabyano-universidad-de-guadalajara",
    project='GSoC26-DeepLense',
    config=config, 
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/leoyan/.netrc.
wandb: Currently logged in as: leofabyano (leofabyano-universidad-de-guadalajara) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [19]:
if run.project_url is not None:
    import webbrowser

    webbrowser.open(run.project_url)

In [ ]:
seed_everything(42)

model = init_model()
criterion = init_criterion()
optimizer = init_optimizer(model.parameters(),
    lr=wandb.config['learning_rate'],
    wd=wandb.config['weight_decay'],)

wandb.watch(model, criterion, log='all', log_freq=10)

In [21]:
# add tag: hide_cell
import traceback
from sklearn.metrics import accuracy_score, f1_score

best_val_loss = float('inf')

try:
    for epoch in range(wandb.config['epochs']):
        log = {
            'epoch': epoch,
            'train/loss': np.nan,
            'train/accuracy': np.nan,
            'train/f1 score': np.nan,
            'val/loss': np.nan,
            'val/accuracy': np.nan,
            'val/f1 score': np.nan,
        }

        # Train 
        model.train()
        loss_train_total, n_batches = 0.0, 0
        all_labels, all_preds = [], []

        for samples, labels in train_loader:
            all_labels.extend(labels.cpu().numpy())
            samples, labels = samples.to(device), labels.to(device)

            samples = transforms(samples)
            pred = model(samples)
            preds = torch.argmax(pred, dim=1)
            all_preds.extend(preds.cpu().numpy())

            loss = criterion(pred, labels)
            n_batches += 1
            loss_train_total += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        log['train/loss'] = loss_train_total / n_batches
        log['train/accuracy'] = accuracy_score(all_labels, all_preds)
        log['train/f1 score'] = f1_score(all_labels, all_preds, average='macro')

        # Validation
        model.eval()
        loss_valid_total, n_batches = 0.0, 0
        all_labels, all_preds = [], []

        with torch.no_grad():
            for samples, labels in valid_loader:
                all_labels.extend(labels.cpu().numpy())
                samples, labels = samples.to(device), labels.to(device)

                pred = model(samples)
                n_batches += 1
                loss_valid_total += criterion(pred, labels).item()
                all_preds.extend(torch.argmax(pred, dim=1).cpu().numpy())

        log['val/loss'] = loss_valid_total / n_batches
        log['val/accuracy'] = accuracy_score(all_labels, all_preds)
        log['val/f1 score'] = f1_score(all_labels, all_preds, average='macro')

        log['lr'] = optimizer.param_groups[0]['lr']

        # Best model checkpoint
        if log['val/loss'] < best_val_loss:
            best_val_loss = log['val/loss']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
                'config': dict(wandb.config),
            }, fr'weights/{WEIGHTS_NAME}-best.pth')
            wandb.save(fr'weights/{WEIGHTS_NAME}-best.pth')

        wandb.log(data=log)

except Exception as e:
    traceback.print_exc()
    wandb.finish()

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


In [22]:
# add tag: hide_cell
import plotly.graph_objects as go

history = wandb.Api().run(f'{run.entity}/{run.project}/{run.id}').history()

fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=('Epoch vs Loss', 'Epoch vs Accuracy', 'Epoch vs F1-Score'),
    horizontal_spacing=0.08
)

fig.add_trace(
    go.Scatter(x=history['epoch'], y=history['train/loss'], mode='lines', 
        name='Train', line=dict(color='blue', dash='solid'), 
        legend='legend'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=history['epoch'], y=history['val/loss'], mode='lines', 
        name='Val', line=dict(color='orange', dash='dash'), 
        legend='legend'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=history['epoch'], y=history['train/accuracy'], mode='lines', 
        name='Train', line=dict(color='green', dash='solid'), 
        legend='legend2'),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=history['epoch'], y=history['val/accuracy'], mode='lines', 
        name='Val', line=dict(color='red', dash='dash'), 
        legend='legend2'),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=history['epoch'], y=history['train/f1 score'], mode='lines', 
        name='Train', line=dict(color='purple', dash='solid'), 
        legend='legend3'),
    row=1, col=3
)
fig.add_trace(
    go.Scatter(x=history['epoch'], y=history['val/f1 score'], mode='lines', 
        name='Val', line=dict(color='brown', dash='dash'), 
        legend='legend3'),
    row=1, col=3
)

fig.update_layout(
    title_text='Model Training Results',
    height=500,
    width=1400,
    
    legend=dict(
        x=0.01, y=0.99, xanchor='left', yanchor='top', 
        bgcolor='rgba(255,255,255,0.7)', bordercolor='lightgrey', borderwidth=1
    ),
    legend2=dict(
        x=0.36, y=0.99, xanchor='left', yanchor='top', 
        bgcolor='rgba(255,255,255,0.7)', bordercolor='lightgrey', borderwidth=1
    ),
    legend3=dict(
        x=0.71, y=0.99, xanchor='left', yanchor='top', 
        bgcolor='rgba(255,255,255,0.7)', bordercolor='lightgrey', borderwidth=1
    )
)

fig.update_xaxes(title_text='Epoch', row=1, col=1)
fig.update_xaxes(title_text='Epoch', row=1, col=2)
fig.update_xaxes(title_text='Epoch', row=1, col=3)

fig.update_yaxes(title_text='Loss', row=1, col=1)
fig.update_yaxes(title_text='Accuracy', row=1, col=2)
fig.update_yaxes(title_text='F1-Score', row=1, col=3)

fig.write_image(format_fig(fig))
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=1\textwidth]{task1_images/model_training_results.png}
\\ Figure 5: Epoch vs Loss, Accuracy, and Macro F1 during the training.
\end{figure}

In [23]:
# add tag: hide_cell
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    #'scheduler_state_dict': scheduler.state_dict(),
    'val_loss': log['val/loss'],
    'config': dict(wandb.config),
}, fr'weights/{WEIGHTS_NAME}-final.pth')
            
artifact = wandb.Artifact(fr'{WEIGHTS_NAME}-best', type='model')
artifact.add_file(fr'weights/{WEIGHTS_NAME}-best.pth')
run.log_artifact(artifact)

<Artifact model-1-best>

# Results

The trained model was evaluated on the held-out validation set using the best checkpoint obtained at epoch 86. Performance is assessed through three complementary diagnostic tools: the classification metrics reported during training (Section 3), the normalized confusion matrix (Figure 6), and the multiclass ROC curve under the One-vs-Rest (OvR) scheme (Figure 7).

The confusion matrix reveals strong classification performance across all three classes, with notably asymmetric behavior. The No Substructure class achieves a recall of 0.99, with marginal confusion distributed equally between the Sphere (0.01) and Vortex (0.00) classes. The Vortex class attains a recall of 0.96, with only minor misclassification into the No Substructure (0.02) and Sphere (0.02) classes. The SubHalo (Sphere) class exhibits the most inter-class confusion, with a recall of 0.86 and a non-negligible misclassification rate into the No Substructure class (0.09) and the Vortex class (0.05). This asymmetry suggests that the morphological features of the Sphere class present a greater discriminative challenge, likely due to its visual similarity with the other two classes under certain lensing configurations.

The ROC curves under the OvR scheme corroborate these findings. All three classes achieve high discriminative power, with AUC scores of 0.9930 (No Substructure), 0.9845 (SubHalo Sphere), and 0.9948 (Vortex). Consistent with the confusion matrix, the Sphere class yields the lowest AUC, though the margin remains small. The mean AUC across all classes is 0.9908, indicating that the model has learned highly separable representations for this three-class gravitational lensing classification task.

In [24]:
for var_name in ['samples', 'labels', 'model', 'criterion', 'optimizer']:
    if var_name in globals():
        del globals()[var_name]

torch.cuda.empty_cache()

In [25]:
# add tag: hide_cell
model = init_model()

checkpoint = torch.load(fr'./weights/{WEIGHTS_NAME}-best.pth', 
    map_location=device, weights_only=True)

model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [26]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

model.eval()

all_labels = [] # to storage true labels 
all_probs = [] # to storage softmax from model output (logits)
all_preds = [] # to storage argmax from softmax output

with torch.no_grad():
    for samples, labels in valid_loader:
        all_labels.extend(labels)
            
        samples = samples.to(device)
        labels = labels.to(device)
                
        pred = model(samples)
        probs = torch.softmax(pred, dim=1) 
        preds = torch.argmax(pred, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

y_true_bin = label_binarize(all_labels, classes=list(range(C_CLASSES)))

fpr = dict() # False Positive Rate
tpr = dict() # True Positive Rate
roc_auc = dict()

for c in range(C_CLASSES):
    fpr[c], tpr[c], _ = roc_curve(y_true_bin[:, c], all_probs[:, c]) # type: ignore
    roc_auc[c] = auc(fpr[c], tpr[c])

In [27]:
# add tag: hide_cell
from sklearn.metrics import confusion_matrix

cm: np.ndarray = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels_name: list = [c.capitalize() for c in valid_loader.dataset.labels_name]

fig = px.imshow(
    cm_normalized, 
    text_auto='.2f', # type: ignore
    x=labels_name, 
    y=labels_name, 
    labels=dict(x='Predicted Class', y='True Class'), 
    title='Confusion Matrix on Validation set',
    height=524, width=524
)

fig.write_image(format_fig(fig))
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=0.8\textwidth]{task1_images/confusion_matrix_on_validation_set.png}
\\ Figure 6: Confusion Matrix on Validation set.
\end{figure}

In [28]:
# add tag: hide_cell
fig = go.Figure()

for i in range(C_CLASSES):
    fig.add_trace(go.Scatter(
        x=fpr[i], y=tpr[i],
        mode='lines',
        name=f'Class {labels_name[i]} (AUC = {roc_auc[i]:.4f})'
    ))
    
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='black'),
    showlegend=False
))

fig.update_layout(
    title='Multiclass ROC Curve (One-vs-Rest)', 
    xaxis_title='False Positive Rate (FPR)', 
    yaxis_title='True Positive Rate (TPR)',
    height=524, width=524,

    legend=dict(
        x=0.98, y=0.02,
        xanchor='right',
        yanchor='bottom',
    )
)

fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1], scaleanchor="x", scaleratio=1)

fig.write_image(format_fig(fig))
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=0.8\textwidth]{task1_images/multiclass_roc_curve_(one-vs-rest).png}
\\ Figure 7: Multiclass ROC curve under the One-vs-Rest (OvR).
\end{figure}

In [29]:
# add tag: hide_cell
wandb.finish()

epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/accuracy,▁▃▅▅▅▆▆▇▇▇▇▇▇▇▇█████████████████████████
train/f1 score,▁▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇████████████████████████
train/loss,█▇▇▆▆▅▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/accuracy,▁▁▁▁▁▅▆▆▇▆▇▇▇▇▇█▇▇█▇▇██▇████████████████
val/f1 score,▂▁▁▅▄▆▇▇▆▆▇▇▇▇▇▇▇▇▇▇████████████████████
val/loss,▄▄▅▅█▆▃▃▃▃▂▃▂▂▂▁▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁
epoch,99
lr,0.0001
train/accuracy,0.94675
